# 05 - NLP Insights
Topic modeling and keyword extraction from termination descriptions and other text fields.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.wf_analysis.nlp.topic_model import TopicModeler
from src.wf_analysis.nlp.keywords import KeywordExtractor

df = pd.read_parquet('data/processed/workforce_clean_base.parquet')
print(f'Total records: {len(df)}')
if 'TerminationDescription' in df.columns:
    n_with_text = df['TerminationDescription'].notna().sum()
    print(f'Records with termination text: {n_with_text} ({n_with_text/len(df):.1%})')

In [ ]:
print('\n=== Topic Modeling ===')
texts = df['TerminationDescription'].dropna().tolist()
if len(texts) > 0:
    tm = TopicModeler(n_topics=5)
    tm.fit(texts)
    topics = tm.transform(texts)
    print(f'Topic distribution:')
    for i, topic in enumerate(tm.get_topics()):
        print(f'  Topic {i}: {" ".join(topic[:8])}')
    print(f'\nTopic assignment counts: {pd.Series(topics).value_counts().to_dict()}')

In [ ]:
print('\n=== Keyword Extraction ===')
if len(texts) > 0:
    ke = KeywordExtractor(max_words=15)
    keywords = ke.extract(' '.join(texts))
    print(f'Top keywords:')
    for kw, score in keywords[:20]:
        print(f'  {kw}: {score:.3f}')

In [ ]:
# Topic distribution visualization
if len(texts) > 0:
    topic_counts = pd.Series(topics).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4))
    topic_counts.plot(kind='bar', ax=ax, title='Topic Distribution in Termination Descriptions')
    ax.set_xlabel('Topic')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()